# Corrective AI — saisonnalité EUR/USD + méta-étiquetage (recherche)

**Hands-On AI Trading**, ch. 08-02 — port [#18901](https://github.com/jsboige/CoursIA/issues/18901).
Ce quantbook est le volet recherche du projet [`main.py`](main.py) : il dérive
la série par fenêtre de la stratégie primaire, réplique le méta-étiquetage
walk-forward multi-seed sur les mêmes données minute, et rend le verdict de la
règle §C (edge ≥ 2σ cross-seed **et** Diebold-Mariano < 0,05).

**Mécanisme du livre** :
1. **Primaire** — saisonnalité intrajournalière EUR/USD (Breedon & Ranaldo 2012) :
   vente 03:00–09:00 ET (heures ouvrées européennes), achat 11:00–15:00 ET
   (heures ouvrées américaines). Le livre mesure Sharpe 0,88 en OOS
   (oct. 2021 – janv. 2023, barres 1 minute EBS).
2. **Corrective AI** — le livre filtre chaque trade via un gradient boosting
   de >100 prédicteurs (API payante predictnow.ai, Sharpe annoncé 1,29). Ici le
   **principe** est porté sans dépendance payante : méta-étiquetage (López de
   Prado 2018) avec ~16 prédicteurs ex-ante, GBM stochastique
   (`subsample=0.7` — la graine pilote le sous-échantillonnage, sinon le
   classifieur est déterministe et le multi-seed n'a aucune variance à mesurer).

**Choix déclarés** : labels sur rendements bruts fenêtre (mid minute, frais
moteur exclus du label, appliqués aux fills réels dans les backtests) ;
fenêtres du lundi au vendredi ; équivalent du warm-up 5 jours = fenêtres à
prédicteurs incomplets écartées.

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime

SEEDS = [0, 1, 7, 42]
WINDOWS = [(3, 9, -1.0), (11, 15, 1.0)]  # (entree ET, sortie ET, cote)
RETRAIN_DAYS, MIN_TRAIN, THRESHOLD = 21, 120, 0.5
ANNUAL_WINDOWS = 2 * 252  # ~2 fenetres/jour ouvre
START, END = datetime(2021, 10, 1), datetime(2023, 1, 31)
print("config: seeds", SEEDS, "| fenetres", WINDOWS,
      "| retrain", RETRAIN_DAYS, "j | min_train", MIN_TRAIN)

config: seeds [0, 1, 7, 42] | fenetres [(3, 9, -1.0), (11, 15, 1.0)] | retrain 21 j | min_train 120


## 1. Données

EUR/USD minute (OANDA) sur la fenêtre du livre. Le flux du quantbook est le
mid des barres — le livre utilise des barres EBS : l'ordre de grandeur du
Sharpe primaire est comparable, pas la décimale.

In [2]:
# Sur le Lab QC, qb est injecte par l'environnement ; sur un kernel
# python3 brut (execution WS), il faut le construire explicitement.
if "qb" not in dir():
    from QuantConnect.Research import QuantBook
    qb = QuantBook()
eurusd = qb.add_forex("EURUSD", Resolution.MINUTE, Market.OANDA)
df = qb.history(eurusd.symbol, START, END, Resolution.MINUTE)
if isinstance(df.index, pd.MultiIndex):
    df = df.loc[eurusd.symbol]
closes = df["close"].dropna()
print(f"barres minute: {len(closes):,} | "
      f"{closes.index[0]} -> {closes.index[-1]}")
closes.tail(3)

barres minute: 494,241 | 2021-10-01 00:01:00 -> 2023-01-31 00:00:00


time
2023-01-30 23:58:00    1.084535
2023-01-30 23:59:00    1.084585
2023-01-31 00:00:00    1.084620
Name: close, dtype: float64

## 2. Stratégie primaire — fenêtres et rendements

Une fenêtre = un trade : entrée au premier bar de l'heure d'ouverture, sortie
au dernier bar avant l'heure de fermeture (miroir des événements planifiés de
`main.py`). `ret_primary` est le rendement signé côté (vente = −Δprix).

In [3]:
rows = []
for d, day in closes.groupby(closes.index.date):
    if d.weekday() > 4:
        continue
    for h_in, h_out, side in WINDOWS:
        w = day.between_time(f"{h_in:02d}:00:00", f"{h_out - 1:02d}:59:00")
        if len(w) < 60:  # fenetre incomplete (jour ferie / gap de flux)
            continue
        rows.append({
            "date": d, "hour": h_in, "side": side,
            "t_entry": w.index[0],
            "ret_primary": (w.iloc[-1] / w.iloc[0] - 1.0) * side,
        })
win = pd.DataFrame(rows)
n = len(win)
mean_r, std_r = win["ret_primary"].mean(), win["ret_primary"].std(ddof=1)
sharpe_primary = mean_r / std_r * np.sqrt(ANNUAL_WINDOWS)
wr = (win["ret_primary"] > 0).mean()
# Convention inverse (papier Breedon-Ranaldo : achat UE / vente US) :
# meme fenetres, cotes opposes -> serie = negation exacte.
win["ret_swap"] = -win["ret_primary"]
sharpe_swap = -sharpe_primary
print(f"fenetres: {n} | win-rate: {wr:.3f} | "
      f"rendement moyen/fenetre: {mean_r * 1e4:.2f} bp")
print(f"Sharpe ann. convention transcription (vente UE): {sharpe_primary:.3f}")
print(f"Sharpe ann. convention papier (achat UE): {sharpe_swap:.3f}")
print(f"reference du livre (OOS, EBS): Sharpe 0.88 | backtest moteur "
      f"correspondant cite dans la PR")
win[["date", "hour", "side", "ret_primary"]].head(6)

fenetres: 690 | win-rate: 0.520 | rendement moyen/fenetre: 0.51 bp
Sharpe ann. convention transcription (vente UE): 0.364
Sharpe ann. convention papier (achat UE): -0.364
reference du livre (OOS, EBS): Sharpe 0.88 | backtest moteur correspondant cite dans la PR


         date  hour  side  ret_primary
0  2021-10-01     3  -1.0    -0.001424
1  2021-10-01    11   1.0     0.000112
2  2021-10-04     3  -1.0    -0.002339
3  2021-10-04    11   1.0     0.000013
4  2021-10-05     3  -1.0    -0.000194
5  2021-10-05    11   1.0     0.000423

## 3. Prédicteurs ex-ante (miroir de `main.py._compute_features`)

10 prédicteurs de marché dérivés des 240 dernières minutes (log-rendements
1/5/15/30/60/120 min, volatilités réalisées 15/30/60 min, drift 240 min) +
6 prédicteurs séquentiels (côté, jour de semaine, win-rates glissants 10/20 de
la primaire, PnL moyen 10, série). Les labels passés ne sont lus qu'après
clôture de leur fenêtre — aucune fuite.

In [4]:
logs = pd.Series(np.log(closes.values), index=closes.index)
lr = logs.diff()
feat = pd.DataFrame(index=closes.index)
for w in (1, 5, 15, 30, 60, 120):
    feat[f"ret{w}"] = logs.diff(w)
for w in (15, 30, 60):
    feat[f"vol{w}"] = lr.rolling(w).std()
feat["ret240"] = logs.diff(239)
feat = feat.dropna()  # equiv. warm-up 5 jours : fenetres sans historique 240 min
print(f"predicteurs de marche: {feat.shape[1]} | barres couvertes: {len(feat):,}")
feat.tail(3)

predicteurs de marche: 10 | barres couvertes: 494,002


                         ret1      ret5     ret15     ret30     ret60  \
time                                                                    
2023-01-30 23:58:00 -0.000023  0.000065  0.000286  0.000304  0.000055   
2023-01-30 23:59:00  0.000046  0.000138  0.000286  0.000401  0.000124   
2023-01-31 00:00:00  0.000032  0.000129  0.000272  0.000512  0.000143   

                       ret120     vol15     vol30     vol60    ret240  
time                                                                   
2023-01-30 23:58:00 -0.000631  0.000041  0.000068  0.000067 -0.000811  
2023-01-30 23:59:00 -0.000553  0.000041  0.000067  0.000067 -0.000742  
2023-01-31 00:00:00 -0.000521  0.000040  0.000065  0.000067 -0.000691  

## 4. Méta-étiquetage walk-forward multi-seed

Boucle strictement chronologique identique à `main.py` : à chaque entrée, le
modèle (entraîné sur les fenêtres **déjà clôturées**, ré-entraîné tous les
21 jours, premier trade après 120 fenêtres) prédit P(fenêtre gagnante) ;
fenêtre prise si p ≥ 0,5. Rendement méta = rendement primaire si pris, 0 sinon.

In [5]:
from sklearn.ensemble import GradientBoostingClassifier

def run_meta(ret_col):
    """Walk-forward miroir de main.py sur la serie de fenetres donnee."""
    per_seed, meta_rets = [], {}
    for seed in SEEDS:
        X_buf, y_buf = [], []
        model, last_fit = None, None
        rets_meta, taken = [], 0
        for _, row in win.iterrows():
            if row["t_entry"] not in feat.index:
                continue
            f_market = feat.loc[row["t_entry"]].to_numpy()
            y_arr = np.array(y_buf)
            win10 = y_arr[-10:].mean() if len(y_arr) else 0.0
            win20 = y_arr[-20:].mean() if len(y_arr) else 0.0
            pnl10 = (2 * win10 - 1) if len(y_arr) else 0.0
            streak, last = 0, None
            for lab in reversed(y_buf):
                if last is None:
                    last = lab
                if lab != last:
                    break
                streak += 1
            feats = np.concatenate([f_market, [
                row["side"], float(row["date"].weekday()),
                win10, win20, pnl10, float(streak)]])
            due = (last_fit is None) or (
                (row["date"] - last_fit).days >= RETRAIN_DAYS)
            if due and len(y_buf) >= MIN_TRAIN:
                model = GradientBoostingClassifier(
                    n_estimators=80, max_depth=3, subsample=0.7,
                    random_state=seed)
                model.fit(X_buf, y_buf)
                last_fit = row["date"]
            take = False
            if model is not None:
                take = model.predict_proba([feats])[0][1] >= THRESHOLD
            if take:
                taken += 1
            rets_meta.append(row[ret_col] if take else 0.0)
            # la fenetre se clot ce jour : son label n'alimente que les suivantes
            X_buf.append(feats)
            y_buf.append(1 if row[ret_col] > 0 else 0)
        m = np.array(rets_meta)
        meta_rets[seed] = m
        per_seed.append({
            "seed": seed,
            "fenetres_evaluees": len(m),
            "prises": taken,
            "sharpe_meta": m.mean() / m.std(ddof=1) * np.sqrt(ANNUAL_WINDOWS),
            "rendement_bp": m.mean() * 1e4,
        })
    return pd.DataFrame(per_seed), meta_rets

runs = {}
for conv, col, base in [("transcription", "ret_primary", sharpe_primary),
                        ("papier", "ret_swap", sharpe_swap)]:
    df_seed, meta_rets = run_meta(col)
    df_seed["sharpe_primaire"] = base
    runs[conv] = {"df": df_seed, "meta": meta_rets, "base": base,
                  "base_series": win[col].to_numpy()[-len(meta_rets[SEEDS[0]]):]}
    print(f"=== convention {conv} (primaire {base:.3f}) ===")
    print(df_seed.to_string(index=False))

=== convention transcription (primaire 0.364) ===
 seed  fenetres_evaluees  prises  sharpe_meta  rendement_bp  sharpe_primaire
    0                689     320     0.674919      0.693756         0.363858
    1                689     310    -0.005232     -0.005160         0.363858
    7                689     306     0.049500      0.048963         0.363858
   42                689     307    -0.023997     -0.024214         0.363858
=== convention papier (primaire -0.364) ===
 seed  fenetres_evaluees  prises  sharpe_meta  rendement_bp  sharpe_primaire
    0                689     260    -0.356503     -0.311843        -0.363858
    1                689     261     0.173919      0.165720        -0.363858
    7                689     276     0.067397      0.064425        -0.363858
   42                689     279    -0.408206     -0.381735        -0.363858


## 5. Validation règle §C — edge cross-seed + Diebold-Mariano

- **edge σ** : (Sharpe méta moyen − Sharpe primaire) / écart-type des Sharpes
  méta inter-seeds — la dispersion mesure la stochasticité du filtre.
- **DM (Ledoit-Wolf 2008)** : test apparié de différence de Sharpe avec SE HAC
  Newey-West (référence dépôt : `m11c_sharpe_test`), appliqué à la série
  seed-ensemble (moyenne inter-seeds des rendements par fenêtre) vs la
  primaire. Unilatéral : H₀ S_méta ≤ S_primaire.
- **Verdict** : `BEATS` exige edge ≥ 2σ **et** p < 0,05 ; `NO BEATS` si l'un
  échoue nettement ; `INCONCLUSIVE` sinon. Jamais « promising ».

In [6]:
from scipy import stats as sps

def _hac_cov(z, q):
    t, k = z.shape
    z = z - z.mean(axis=0, keepdims=True)
    omega = (z.T @ z) / t
    for h in range(1, q + 1):
        w = 1.0 - h / (q + 1.0)
        gh = (z[h:].T @ z[:-h]) / t
        omega += w * (gh + gh.T)
    return omega

def ledoit_wolf_sharpe_diff_se(r_a, r_b):
    r_a = np.asarray(r_a, float).ravel()
    r_b = np.asarray(r_b, float).ravel()
    t = len(r_a)
    if t < 30 or len(r_a) != len(r_b):
        return (np.nan,) * 4
    mu_a, mu_b = r_a.mean(), r_b.mean()
    s2a, s2b = r_a.var(), r_b.var()
    sa, sb = np.sqrt(s2a), np.sqrt(s2b)
    if sa < 1e-12 or sb < 1e-12:
        return (np.nan,) * 4
    ua, ub = r_a - mu_a, r_b - mu_b
    z = np.column_stack([ua, ua * ua - s2a, ub, ub * ub - s2b])
    q = max(1, int(np.floor(t ** 0.25)))
    omega = _hac_cov(z, q)
    grad = np.array([1 / sa, -mu_a / (2 * sa ** 3), -1 / sb, mu_b / (2 * sb ** 3)])
    se = float(np.sqrt(grad @ omega @ grad / t))
    return mu_a / sa, mu_b / sb, mu_a / sa - mu_b / sb, se

verdicts = {}
for conv in runs:
    r = runs[conv]
    meta_sharpes = r["df"]["sharpe_meta"].to_numpy()
    mean_meta, std_meta = meta_sharpes.mean(), meta_sharpes.std(ddof=1)
    edge_sigma = ((mean_meta - r["base"]) / std_meta
                  if std_meta > 1e-12 else np.inf)
    ens = np.mean([r["meta"][s_] for s_ in SEEDS], axis=0)
    primary_matched = win["ret_primary" if conv == "transcription"
                          else "ret_swap"].to_numpy()[-len(ens):]
    sa_, sb_, sdiff, se = ledoit_wolf_sharpe_diff_se(ens, primary_matched)
    t_stat = sdiff / se if se and se == se else np.nan
    p_one = 1 - sps.norm.cdf(t_stat) if t_stat == t_stat else np.nan
    verdicts[conv] = ("BEATS" if (edge_sigma >= 2.0 and p_one < 0.05) else
                      ("NO BEATS" if (mean_meta <= r["base"] or p_one >= 0.05)
                       else "INCONCLUSIVE"))
    print(f"=== convention {conv} ===")
    print(f"Sharpe primaire: {r['base']:.3f} | meta par seed: "
          + ", ".join(f"{v:.3f}" for v in meta_sharpes))
    print(f"moyenne meta: {mean_meta:.3f} +/- {std_meta:.3f} | "
          f"edge = {edge_sigma:.2f} sigma | DM t={t_stat:.3f} "
          f"p_unilat={p_one:.4f}")
    print(f"VERDICT regle C: {verdicts[conv]}")

=== convention transcription ===
Sharpe primaire: 0.364 | meta par seed: 0.675, -0.005, 0.049, -0.024
moyenne meta: 0.174 +/- 0.336 | edge = -0.57 sigma | DM t=-0.334 p_unilat=0.6310
VERDICT regle C: NO BEATS
=== convention papier ===
Sharpe primaire: -0.364 | meta par seed: -0.357, 0.174, 0.067, -0.408
moyenne meta: -0.131 +/- 0.294 | edge = 0.79 sigma | DM t=0.375 p_unilat=0.3539
VERDICT regle C: NO BEATS


## 6. Rapport de biais (règle §C-7)

Un edge porté par le biais (rendement moyen signé) plutôt que par la précision
(win-rate / ratio gain-perte) se déclare comme tel.

In [7]:
def leg_stats(r):
    r = np.asarray(r, float)
    wins, losses = r[r > 0], r[r < 0]
    return {
        "rendement_moyen_bp": r.mean() * 1e4,
        "win_rate": (r > 0).mean() if len(r) else np.nan,
        "ratio_gain_perte": (wins.mean() / abs(losses.mean())
                             if len(wins) and len(losses) else np.nan),
        "ecart_type_bp": r.std(ddof=1) * 1e4,
    }

for conv in runs:
    r = runs[conv]
    rows_bias = [{"jambe": f"{conv} / primaire",
                  **leg_stats(r["base_series"])}]
    for seed in SEEDS:
        m = r["meta"][seed]
        taken_mask = m != 0.0
        rows_bias.append({
            "jambe": f"{conv} / meta-s{seed} (prises)",
            **leg_stats(m[taken_mask]),
        })
    print(f"=== biais convention {conv} ===")
    print(pd.DataFrame(rows_bias).to_string(index=False))
print("\nlecture: si le Sharpe meta monte sans que le win-rate des fenetres "
      "prises ne monte, l'edge est un artefact de selection, pas de precision.")

=== biais convention transcription ===
                            jambe  rendement_moyen_bp  win_rate  ratio_gain_perte  ecart_type_bp
         transcription / primaire            0.528863  0.521045          0.962403      31.327894
 transcription / meta-s0 (prises)            1.493744  0.534375          0.980623      33.872119
 transcription / meta-s1 (prises)           -0.011468  0.516129          0.936638      33.033489
 transcription / meta-s7 (prises)            0.110248  0.526144          0.908623      33.352305
transcription / meta-s42 (prises)           -0.054343  0.514658          0.939031      33.966191
=== biais convention papier ===
                     jambe  rendement_moyen_bp  win_rate  ratio_gain_perte  ecart_type_bp
         papier / primaire           -0.528863  0.478955          1.039066      31.327894
 papier / meta-s0 (prises)           -0.826384  0.511538          0.890158      31.999422
 papier / meta-s1 (prises)            0.437476  0.505747          1.011242   

## 7. Limites et robustesse

- **Fenêtre longue 2018-2026** : la robustesse hors fenêtre du livre est mesurée
  par les backtests moteur (frais réels, fills, equity curve) — matrice
  `long-*` rapportée dans le corps de la PR de livraison.
- **Écarts déclarés au livre** : flux mid OANDA vs barres EBS ; ~16 prédicteurs
  vs >100 (predictnow.ai, payant) ; fenêtre minute ≠ barres du livre exactes.
- Le Sharpe « par fenêtre » de ce notebook n'est pas le Sharpe equity-curve
  des backtests : bases différentes, mêmes ordres de grandeur attendus.